<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Import Libraries</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Import Libraries
    </h1>
</div>


In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from config.spark_config import SparkConfig
from app.platform_app import PlatformApp
from utils.logger import LoggerConfig

<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Set up</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Set up
    </h1>
</div>


In [2]:
logger = LoggerConfig().setup_logger(log_dir=None)
spark = SparkConfig.create_spark(app_name="FMCG Domain", logger=logger, use_databricks=True)
app = PlatformApp(spark=spark, logger=logger, catalog_name="fmcg_domain")

2026-03-24 19:09:45 | INFO     | Logging configured: level=DEBUG, format=colored
2026-03-24 19:09:47 | INFO     | Connected to Databricks via Spark Connect.
2026-03-24 19:09:47 | INFO     | Initializing Data Platform...
2026-03-24 19:09:47 | INFO     | Spark session initialized


In [ ]:
spark.sql("""
CREATE VIEW IF NOT EXISTS fmcg_domain.gold.vw_fact_orders_enriched AS
SELECT 
    fo.date,
    fo.product_code,
    fo.customer_code,

    -- Date attributes
    dd.date_key,
    dd.year,
    dd.month_name,
    dd.month_short_name,
    dd.quarter,
    dd.year_quarter,

    -- Customer attributes
    dc.customer,
    dc.market,
    dc.platform,
    dc.channel,

    -- Product attributes
    dp.division,
    dp.category,
    dp.product,
    dp.variant,

    -- Metrics
    fo.sold_quantity,
    gp.price_inr,

    -- Derived Metric
    (fo.sold_quantity * gp.price_inr) AS total_amount_inr

FROM fmcg_domain.gold.fact_orders fo

LEFT JOIN fmcg_domain.gold.dim_date dd
       ON fo.date = dd.date

LEFT JOIN fmcg_domain.gold.dim_customers dc
       ON fo.customer_code = dc.customer_code

LEFT JOIN fmcg_domain.gold.dim_products dp
       ON fo.product_code = dp.product_code

LEFT JOIN fmcg_domain.gold.dim_gross_price gp
       ON fo.product_code = gp.product_code
      AND YEAR(fo.date) = gp.year
""")

""


In [5]:
app.stop()

2026-03-24 19:17:45 | INFO     | Stopping Spark session...
2026-03-24 19:17:46 | INFO     | Spark stopped.
